# Example Notebook 04:

Fine-tuning of TimeXer model with Composite Losses

In [ ]:
%load_ext autoreload
%autoreload 2

from lightning import Trainer
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger
import mlflow
import torch

from src import BESSTimeXer, TimeXerDataModule
import config

In [ ]:
# prepare data
data = TimeXerDataModule(**config.DATA_CONFIG, battery_config=config.BATTERY_CONFIG)

To initialize ML Flow for experiment tracking, run in terminal:
```
mlflow ui
```

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
EXPERIMENT_NAME = "BESSTimeXer-Base"
CONFIG = config.BATTERY_CONFIG | config.DATA_CONFIG | config.MODEL_CONFIG

In [ ]:
# Load and prepare model weights and checkpoint from pre-trained model
checkpoint = torch.load("2/707bd0fad77d4970ada59f4f3daf786a/checkpoints/epoch=7-step=8344.ckpt", map_location="cpu")
completed_epochs = checkpoint["loops"]["fit_loop"]["epoch_progress"]["current"]["completed"]
if "optimizer_states" in checkpoint:
    checkpoint["optimizer_states"] = []

# Save prepared checkpoint for fine-tuning
torch.save(checkpoint, "weights/finetune_checkpoint.ckpt")

In [ ]:
# Set up Model for fine-tuning
model = BESSTimeXer.load_from_checkpoint(checkpoint_path="weights/finetune_checkpoint.ckpt", **config.MODEL_CONFIG, scaler=data.scaler)
penalty_config = config.SPOPLUS_CONFIG
CONFIG = CONFIG | penalty_config
model.set_phase('finetune', **penalty_config)

In [ ]:
# Set up logging with MLFlow
RUN_NAME = f"{config.MODEL_CONFIG['loss']}_{penalty_config['penalty']}_{penalty_config['penalty_lambda']}"
mlflow_logger_ft = MLFlowLogger(
    experiment_name=EXPERIMENT_NAME,
    run_name=RUN_NAME,
    tracking_uri=mlflow.get_tracking_uri(),
    log_model=True
)

# Log hyperparameters for this specific run variant
mlflow_logger_ft.log_hyperparams(CONFIG)

# Set up Trainer
finetune_trainer = Trainer(
    callbacks=[
       EarlyStopping(
           monitor=f"val_loss",
           patience=3,
           min_delta=0.005,
           verbose=True,
           mode="min",
       ),
    ],
    gradient_clip_val=0.5,
    max_epochs=5,
    logger=mlflow_logger_ft
)

#finetune_trainer.test(model, datamodule=data)

In [ ]:
# Fine tune model
finetune_trainer.fit(model, datamodule=data)

In [ ]:
finetune_trainer.test(model, datamodule=data)